# Install databricks packages

In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

# Load Docs and enable Data Feed

In [0]:
docs_table = "workspace.default.facility_documents"

spark.sql(f"ALTER TABLE {docs_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

display(spark.sql(f"SELECT * FROM {docs_table} LIMIT 5"))

# Create Vector Search Endpoint

In [0]:
from databricks.vector_search.client import VectorSearchClient

vector_search_endpoint = "vector_search_facilities_endpoint"

vsc = VectorSearchClient()

existing = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]

if vector_search_endpoint not in existing:
    vsc.create_endpoint(
        name=vector_search_endpoint,
        endpoint_type="STANDARD"
    )
    print(f"Creating endpoint '{vector_search_endpoint}'...")
else:
    print(f"Endpoint '{vector_search_endpoint}' already exists.")


# Create Vector Search Index

In [0]:
from databricks.vector_search.client import VectorSearchClient

index_name = "workspace.default.facility_documents_index"

vsc = VectorSearchClient(disable_notice=True)

index = vsc.create_delta_sync_index(
  endpoint_name=vector_search_endpoint,
  source_table_name=docs_table,
  index_name=index_name,
  pipeline_type="TRIGGERED",
  primary_key="id",
  embedding_source_column="document",
  embedding_model_endpoint_name="databricks-gte-large-en",
  columns_to_sync = [
    "id","document","name","organization_type","facility_type","address_city",
    "address_stateOrRegion","address_country","lat","lon","numberDoctors","capacity"
    ]
)

print(f"index '{index_name}' created for table '{docs_table}' using endpoint '{vector_search_endpoint}'")

In [0]:
index = vsc.get_index(vector_search_endpoint, index_name)
index.sync()
print("Sync triggered — embeddings being generated for all rows...")

# Test Similarity Search

In [0]:
results = index.similarity_search(
    query_text  = "What services does 2BN Military Hospital offer?",
    columns     = ["id", "name", "document"],
    num_results = 5
)

print(results)